# 02 — Silver Validation

**Purpose:** Typed clean validated Silver tables.  
**Runtime:** Microsoft Fabric PySpark with attached default Lakehouse.  
**Status:** Provided as source; requires execution and verification in your Fabric tenant.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

# Assumption: source files are snapshot batches, not a CDC stream.
# Dedupe on business keys is deterministic for injected exact duplicates.
c=(spark.table('bronze_customers').select('customer_id','customer_name','region','signup_date')
   .dropDuplicates(['customer_id'])
   .withColumn('region',F.when(F.col('region').isNull() | (F.trim('region')==''),'Unknown').otherwise(F.col('region')))
   .withColumn('signup_date',F.to_date('signup_date'))
   .filter(F.col('customer_id').isNotNull() & F.col('signup_date').isNotNull()))
p=(spark.table('bronze_products').select('product_id','product_name','category','list_price')
   .dropDuplicates(['product_id'])
   .withColumn('list_price',F.col('list_price').cast('double'))
   .filter(F.col('product_id').isNotNull() & (F.col('list_price')>0)))
o=(spark.table('bronze_orders').select('order_id','customer_id','order_date','order_status')
   .dropDuplicates(['order_id'])
   .withColumn('order_date',F.to_date('order_date'))
   .filter(F.col('order_id').isNotNull() & F.col('order_date').isNotNull())
   .join(c.select('customer_id'),on='customer_id',how='left_semi'))
i=(spark.table('bronze_order_items')
   .select('order_item_id','order_id','product_id','quantity','unit_price','discount_pct')
   .dropDuplicates(['order_item_id'])
   .withColumn('quantity',F.col('quantity').cast('int'))
   .withColumn('unit_price',F.col('unit_price').cast('double'))
   .withColumn('discount_pct',F.col('discount_pct').cast('double'))
   .filter((F.col('quantity')>0)&(F.col('unit_price')>=0)&F.col('discount_pct').between(0,1))
   .join(o.select('order_id'),on='order_id',how='left_semi')
   .join(p.select('product_id'),on='product_id',how='left_semi')
   .withColumn('gross_sales',F.round(F.col('quantity')*F.col('unit_price'),2))
   .withColumn('net_sales',F.round(F.col('gross_sales')*(1-F.col('discount_pct')),2)))
for name,df in {'customers':c,'products':p,'orders':o,'order_items':i}.items():
    (df.write.format('delta').mode('overwrite').option('overwriteSchema','true')
        .saveAsTable('silver_'+name))
    print(f'silver_{name}: {df.count():,} valid rows')
print('Rows rejected from order_items:', spark.table('bronze_order_items').count()-i.count())
